# مسئلهٔ ۱ — مدل اولیهٔ تشخیص تصادف با ResNet18

مدل برای هر فریم احتمال تصادف را پیش‌بینی می‌کند. سپس احتمال‌های ۸ فریم هر ویدئو میانگین گرفته می‌شوند تا تصمیم نهایی در سطح ویدئو ساخته شود.

> این یک baseline سبک و قابل توضیح است. در مرحله‌های بعد می‌توانیم مدل دنباله‌ای اضافه کنیم تا ترتیب زمانی فریم‌ها را نیز یاد بگیرد.

In [8]:
# این سلول را فقط یک‌بار اجرا کنید؛ سپس در صورت درخواست VS Code، kernel را restart کنید.
%pip install torch torchvision

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Note: you may need to restart the kernel to use updated packages.


In [9]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, f1_score
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ResNet18_Weights, resnet18
from torchvision.transforms import v2

DATA_ROOT = Path(r'P:\NexarCollisionData')
FRAME_SPLITS_PATH = DATA_ROOT / 'frame_splits.csv'
MODEL_DIR = DATA_ROOT / 'models'
MODEL_DIR.mkdir(exist_ok=True)

RANDOM_SEED = 42
BATCH_SIZE = 32
EPOCHS = 3
LEARNING_RATE = 1e-4
NUM_WORKERS = 0  # روی Windows/Jupyter پایدارتر است.

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


In [10]:
frame_splits = pd.read_csv(FRAME_SPLITS_PATH)
train_frames = frame_splits.query("split == 'train'").reset_index(drop=True)
val_frames = frame_splits.query("split == 'validation'").reset_index(drop=True)

assert len(train_frames) == 3840
assert len(val_frames) == 960
assert train_frames['frame_path'].map(lambda path: Path(path).exists()).all()
assert val_frames['frame_path'].map(lambda path: Path(path).exists()).all()
print('Train frames:', len(train_frames), ' | Validation frames:', len(val_frames))

Train frames: 3840  | Validation frames: 960


In [11]:
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = v2.Compose([
    v2.ToImage(),
    v2.RandomHorizontalFlip(p=0.5),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=imagenet_mean, std=imagenet_std),
])
validation_transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=imagenet_mean, std=imagenet_std),
])

class FrameDataset(Dataset):
    def __init__(self, table, transform):
        self.table = table.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.table)

    def __getitem__(self, index):
        row = self.table.iloc[index]
        image = Image.open(row.frame_path).convert('RGB')
        return self.transform(image), int(row.label), row.video_id

train_loader = DataLoader(
    FrameDataset(train_frames, train_transform), batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=(device.type == 'cuda'),
)
val_loader = DataLoader(
    FrameDataset(val_frames, validation_transform), batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(device.type == 'cuda'),
)

In [12]:
weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)

# برای baseline سبک، فقط آخرین block و head را fine-tune می‌کنیم.
for parameter in model.parameters():
    parameter.requires_grad = False
for parameter in model.layer4.parameters():
    parameter.requires_grad = True
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    (parameter for parameter in model.parameters() if parameter.requires_grad),
    lr=LEARNING_RATE, weight_decay=1e-4,
)
sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

8394754

In [13]:
def evaluate_video_level(model, loader):
    model.eval()
    records = []
    with torch.no_grad():
        for images, labels, video_ids in loader:
            probabilities = torch.softmax(model(images.to(device)), dim=1)[:, 1].cpu().numpy()
            for video_id, label, probability in zip(video_ids, labels.numpy(), probabilities):
                # DataLoader شناسه‌ی عددی را Tensor می‌کند؛ برای گروه‌بندی صحیح 8 فریم، آن را به int تبدیل می‌کنیم.
                records.append({'video_id': int(video_id), 'label': int(label), 'positive_probability': float(probability)})

    frame_predictions = pd.DataFrame(records)
    video_predictions = frame_predictions.groupby('video_id', as_index=False).agg(
        label=('label', 'first'),
        positive_probability=('positive_probability', 'mean'),
    )
    video_predictions['prediction'] = (video_predictions['positive_probability'] >= 0.5).astype(int)
    return video_predictions

best_f1 = -1.0
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for images, labels, _ in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)

    validation_predictions = evaluate_video_level(model, val_loader)
    validation_f1 = f1_score(validation_predictions.label, validation_predictions.prediction)
    validation_accuracy = accuracy_score(validation_predictions.label, validation_predictions.prediction)
    metrics = {
        'epoch': epoch, 'train_loss': total_loss / len(train_loader.dataset),
        'validation_f1': validation_f1, 'validation_accuracy': validation_accuracy,
    }
    history.append(metrics)
    print(metrics)

    if validation_f1 > best_f1:
        best_f1 = validation_f1
        torch.save({
            'model_state_dict': model.state_dict(),
            'epoch': epoch,
            'validation_f1': validation_f1,
        }, MODEL_DIR / 'resnet18_frame_baseline.pt')

pd.DataFrame(history).to_csv(MODEL_DIR / 'resnet18_training_history.csv', index=False)

{'epoch': 1, 'train_loss': 0.5218212746083737, 'validation_f1': 0.6349206349206349, 'validation_accuracy': 0.6166666666666667}
{'epoch': 2, 'train_loss': 0.21682308856397867, 'validation_f1': 0.6814814814814815, 'validation_accuracy': 0.6416666666666667}
{'epoch': 3, 'train_loss': 0.0903144706816723, 'validation_f1': 0.631578947368421, 'validation_accuracy': 0.65}


In [14]:
checkpoint = torch.load(MODEL_DIR / 'resnet18_frame_baseline.pt', map_location=device, weights_only=True)
model.load_state_dict(checkpoint['model_state_dict'])
validation_predictions = evaluate_video_level(model, val_loader)
validation_predictions.to_csv(MODEL_DIR / 'resnet18_validation_predictions.csv', index=False)

print('Best validation F1:', f"{checkpoint['validation_f1']:.4f}")
print(classification_report(validation_predictions.label, validation_predictions.prediction, digits=4))
validation_predictions.head()

Best validation F1: 0.6815
              precision    recall  f1-score   support

           0     0.6889    0.5167    0.5905        60
           1     0.6133    0.7667    0.6815        60

    accuracy                         0.6417       120
   macro avg     0.6511    0.6417    0.6360       120
weighted avg     0.6511    0.6417    0.6360       120



,video_id,label,positive_probability,prediction
0,14,1,0.542827,1
1,29,1,0.274635,0
2,31,1,0.763241,1
3,32,1,0.880159,1
4,56,1,0.703052,1


## خروجی‌های مدل

- `models/resnet18_frame_baseline.pt`: بهترین وزن مدل براساس F1 اعتبارسنجی در سطح ویدئو.
- `models/resnet18_training_history.csv`: روند loss و معیارهای validation.
- `models/resnet18_validation_predictions.csv`: احتمال و پیش‌بینی هر ویدئوی validation.

پس از ثبت نتیجهٔ baseline، می‌توانیم ResNet18 را با یک LSTM یا Transformer زمانی برای استفاده از ترتیب ۸ فریم توسعه دهیم.